# Output Parser
**Output Parser**는 대규모 언어 모델(LLM, Large Language Model)의 출력 결과를 **애플리케이션에서 활용할 수 있도록 적절한 형식으로 변환**하는 도구이다.
- LLM은 일반적으로 텍스트 형태로 응답을 생성하지만, 이 텍스트는 그대로 활용하기 어려운 경우가 많다.
- Output Parser는 이러한 **비구조적 텍스트 데이터를 구조화된 데이터로 변환**하여 프로그램에서 활용 가능하도록 만든다.
- 예를 들어, 키워드 리스트를 뽑거나 JSON 형식으로 정보를 변환하는 데 사용된다.

## 주요 Output Parser 종류

1. **CommaSeparatedListOutputParser**
   - 쉼표로 구분된 텍스트를 파싱하여 리스트 형태로 변환한다.
   - 예: `"사과, 바나나, 포도"` → `["사과", "바나나", "포도"]`
2. **JsonOutputParser**
   - LLM의 출력이 JSON 형식일 때 이를 Python의 `dict` 객체로 변환한다.
   - JSON(JavaScript Object Notation)은 데이터 구조를 표현하기 위한 경량 포맷이다.
3. **PydanticOutputParser**
   - JSON 데이터를 Python의 [Pydantic](https://docs.pydantic.dev) 모델로 변환한다.
   - Pydantic은 데이터 유효성 검사와 설정 관리에 널리 사용되는 Python 라이브러리이다.
4. **StrOutputParser**
   - 모델의 출력 결과를 단순 문자열로 반환한다.
   - Chat 기반 모델은 Message 객체의 속성으로 LLM 결과를 반환한다. 거기에서 응답 문자열만 추출해서 반환한다.
> `JsonOutputParser`, `PydanticOutputParser` 는 모두 Pydantic을 사용해 데이터 구조(schema)를 정의하고, 해당 구조에 따라 출력을 검증하고 변환한다.

## 주요 메소드
- `parse(text: str)`
  - LLM이 생성한 문자열 응답을 받아 정해진 구조로 변환하여 반환한다.
- `get_format_instructions() -> str`
  - 각 OutputParer가 변환할 수있는 형식으로 LLM이 응답하도록 하는 프롬프트 텍스트를 반환한다.
  - 이 내용을 프롬프트에 넣어서 LLM이 정확한 포맷으로 응답하도록 유도한다.
  
## 참고
- Output Parser는 일반적으로 [`Runnable`](05_chaing_LECL.ipynb#Runnable) 인터페이스를 상속하여 구현되며, `invoke()` 메서드를 통해 실행할 수 있다.
- `invoke()`는 내부적으로 `parse()`를 호출하여 동작한다.
- 필요한 경우 Output Parser를 직접 구현하여 사용자 정의 출력 포맷을 처리할 수도 있다. 


## StrOutputParser
- 모델(LLM)의 출력 결과를 string으로 변환하여 반환하는 output parser.
- Chat Model은  Message 객체에서 content 속성값을 추출하여 문자열로 반환한다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
prompt = ChatPromptTemplate.from_template(
    template="한국의 {topic}에 관련된 속담 {count}개를 알려줘. 네가 만들지 말고 실제 있는 속담을 알려줘"
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

# prompt-[Query]->model-[response]->parser-[문자열]->최종결과
query = prompt.invoke({"topic":"호랑이", "count":3})
print(query)
response = model.invoke(query)
response

messages=[HumanMessage(content='한국의 호랑이에 관련된 속담 3개를 알려줘. 네가 만들지 말고 실제 있는 속담을 알려줘', additional_kwargs={}, response_metadata={})]


AIMessage(content='다음은 **한국에서 실제로 쓰이는(확인되는) 호랑이 관련 속담/관용구** 3개입니다.\n\n1. **호랑이도 제 말 하면 온다**  \n   - “말을 꺼내면(언급하면) 그 일이 생기거나 해당 인물이 나타난다”는 뜻.\n\n2. **호랑이 굴에 가야 호랑이 새끼 잡는다**  \n   - “위험을 감수해야 큰 성과를 얻는다”는 의미.\n\n3. **호랑이에게 물려 가도 정신만 차리면 산다**  \n   - “큰 위기 속에서도 정신을 잃지 않으면 벗어날 수 있다”는 뜻.\n\n원하시면, 위 속담들의 **뜻/유사 속담/사용 예시**도 함께 정리해드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 191, 'prompt_tokens': 35, 'total_tokens': 226, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Dto0hapPeHUhqYQifELKQlhloSSQj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ef2f5-ba33-7730-855c-2ded8e6d5c87-0', tool_calls=[], invalid_tool_calls=[], usage_metadata=

In [12]:
print(response.content)

다음은 **한국에서 실제로 쓰이는(확인되는) 호랑이 관련 속담/관용구** 3개입니다.

1. **호랑이도 제 말 하면 온다**  
   - “말을 꺼내면(언급하면) 그 일이 생기거나 해당 인물이 나타난다”는 뜻.

2. **호랑이 굴에 가야 호랑이 새끼 잡는다**  
   - “위험을 감수해야 큰 성과를 얻는다”는 의미.

3. **호랑이에게 물려 가도 정신만 차리면 산다**  
   - “큰 위기 속에서도 정신을 잃지 않으면 벗어날 수 있다”는 뜻.

원하시면, 위 속담들의 **뜻/유사 속담/사용 예시**도 함께 정리해드릴게요.


In [13]:
res_str = parser.invoke(response)   # .content를 하지 않아도 문자만 뽑는다
print(res_str)

다음은 **한국에서 실제로 쓰이는(확인되는) 호랑이 관련 속담/관용구** 3개입니다.

1. **호랑이도 제 말 하면 온다**  
   - “말을 꺼내면(언급하면) 그 일이 생기거나 해당 인물이 나타난다”는 뜻.

2. **호랑이 굴에 가야 호랑이 새끼 잡는다**  
   - “위험을 감수해야 큰 성과를 얻는다”는 의미.

3. **호랑이에게 물려 가도 정신만 차리면 산다**  
   - “큰 위기 속에서도 정신을 잃지 않으면 벗어날 수 있다”는 뜻.

원하시면, 위 속담들의 **뜻/유사 속담/사용 예시**도 함께 정리해드릴게요.


## CommaSeparatedListOutputParser

- 쉼표로 구분된 텍스트를 파싱하여 리스트 형태로 변환한다.
  - "a,b,c" => ['a','b','c']

In [17]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
txt = "서울,인천,부산,광주,대구,대전,울산"
# txt = "찾은 도시이름은 서울과 인천과 대전입니다."

r1 = parser.parse(txt)
r2 = parser.invoke(txt)

print(r1)
print(r2)

['서울', '인천', '부산', '광주', '대구', '대전', '울산']
['서울', '인천', '부산', '광주', '대구', '대전', '울산']


In [18]:
# Output Parser에 맞는 출력을 LLM 모델이 하도록 프롬프트에 넣을 지침을 조회.
instruction = parser.get_format_instructions()
print(instruction)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [21]:
prompt = ChatPromptTemplate(
    messages=[
        {"role":"system", "content":"출력형식: {output_format}"},
        {"role":"user", "content":"{query}"}
    ],
    partial_variables={"output_format":instruction} # input_variables에 값을 프롬프트템플릿 생성하면서 입력한다.
)
model = ChatOpenAI(model="gpt-5.4-nano")

query = prompt.invoke({"query":"자동차 종류 다섯가지를 알려줘."})
response = model.invoke(query)

In [24]:
res = parser.invoke(response)
print(res)

['세단', 'SUV', '해치백', '쿠페', '밴']


## JsonOutputParser

- JSON 형식의 응답을 dictionary로 반환한다.
- JSON 형식을 정하려는 경우 [Pydantic](Ref_typing_Pydantic.ipynb)을 이용해 JSON 스키마를 정의하여 JsonOutputParser 생성시 전달한다.
  - Pydantic 모델클래스를 이용해 LLM 모델이 응답할 때 json의 어떤 key에 어떤 응답을 작성할 지 Field로 정의한다.
  - Schema 지정은 필수는 아니다. 
- LLM이 JSON Schema를 따르는 형태로 응답을 하면 JsonOutputParser는 Dictionary로 변환한다.

In [27]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
# LLM 출력 형식 프롬프트
print(parser.get_format_instructions())

txt = '{"name":"이순신", "age":30, "address":"서울"}'
r1 = parser.parse(txt)
r2 = parser.invoke(txt)

print(type(r1), r1)
print(type(r2), r2)

Return a JSON object.
<class 'dict'> {'name': '이순신', 'age': 30, 'address': '서울'}
<class 'dict'> {'name': '이순신', 'age': 30, 'address': '서울'}


In [40]:
parser = JsonOutputParser()
prompt = ChatPromptTemplate(
    messages=[
        ("system","output format: {output_format}"),
        ("user", "{query}")
    ],
    partial_variables = {"output_format":parser.get_format_instructions()}
)
model = ChatOpenAI(model="gpt-5.4-mini")

query = prompt.invoke({"query":"이순신 장군에 대해서 알려줘."})
# query.messages
res = model.invoke(query)

In [47]:
print(res.content)
# res.content
final_answer = parser.invoke(res)

{"answer":"이순신(1545~1598)은 조선 중기의 무신이자 임진왜란 때 조선을 지켜낸 대표적인 장군입니다. 한국 역사에서 가장 존경받는 인물 중 한 명으로 꼽힙니다.\n\n핵심만 정리하면:\n- 출생: 1545년 한성(서울) 출생\n- 직업: 조선의 무신, 수군통제사\n- 대표 업적: 임진왜란(1592~1598)에서 조선 수군을 이끌고 일본군의 해상 보급로를 차단해 전쟁의 흐름을 바꿈\n- 유명한 승리: 옥포해전, 사천해전, 한산도대첩, 명량해전, 노량해전\n- 대표 무기/전술: 거북선 활용, 학익진 전법\n- 최후: 1598년 노량해전에서 전사\n\n왜 유명한가?\n- 적은 병력과 불리한 조건에서도 뛰어난 전략과 지휘로 연전연승을 거뒀습니다.\n- 특히 명량해전에서는 매우 적은 수의 배로 대규모 일본 수군을 막아내며 큰 승리를 거뒀습니다.\n- 그의 업적은 단순한 전투 승리를 넘어, 조선이 전쟁에서 버틸 수 있는 결정적 기반을 마련한 것으로 평가됩니다.\n\n관련 기록\n- 이순신은 직접 전쟁 상황과 생각을 기록한 「난중일기」를 남겼습니다.\n- 그의 인간적인 면모, 고민, 책임감이 잘 드러나는 중요한 역사 자료입니다.\n\n원하시면 다음 중 하나로 더 자세히 설명해드릴게요:\n1. 이순신의 생애\n2. 주요 해전 정리\n3. 명량해전이 왜 대단한지\n4. 난중일기 소개\n5. 어린이용으로 쉽게 설명"}


In [48]:
final_answer['answer']
final_answer.keys()

dict_keys(['answer'])

In [51]:
# JSON 형식에 대한 스키마 설계
## 어떤 키를 가지며, 그 키에 어떤 값을 넣어야 하는지를 설계. => pydantic 모델을 이용
from pydantic import BaseModel, Field
class PersonInfoSchema(BaseModel):
    """ 
    인물 정보에 대한 응답을 위한 Schema
    """

    # 변수명(key): 결과값의 타입 = Field(description="어떤 값을 넣을지 설명")
    name: str = Field(description="조회한 사람의 이름")
    yob: int = Field(description="조회한 사람의 출생 년도. 모를 경우는 -1을 대입")
    yod: int = Field(description="조회한 사람의 사망 년도. 모를 경우는 -1을 대입")
    profile: str = Field(description="조회한 사람의 주요 업적 소개")

parser2 = JsonOutputParser(pydantic_object=PersonInfoSchema)
print(parser2.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [59]:
prompt2 = ChatPromptTemplate(
    messages=[
        ("system","output format: {output_format}"),
        ("user", "{query}")
    ],
    partial_variables = {"output_format":parser2.get_format_instructions()}
)

query2 = prompt2.invoke({"query":"을지문덕 장군에 대해 알려줘."})
res2 = model.invoke(query2)

In [60]:
# print(res2.content)
answer = parser2.invoke(res2)
answer#['name']

{'name': '을지문덕',
 'yob': -1,
 'yod': -1,
 'profile': '고구려의 명장으로, 7세기 수나라의 대규모 침공을 막아낸 인물이다. 특히 살수대첩을 통해 수나라 군대를 크게 격파한 것으로 유명하며, 뛰어난 지략과 전략으로 고구려의 국방을 지킨 대표적 영웅으로 평가된다.'}

## PydanticOutputParser

- JSON 형태로 받은 응답을 Pydantic 모델로 변환하여 반환한다.
- 구현은 JsonOutputParser와 동일한데 parsing 결과를 pydantic 모델타입으로 반환한다.

In [1]:
from pydantic import BaseModel, Field
class PersonInfoSchema(BaseModel):
    """ 
    인물 정보에 대한 응답을 위한 Schema
    """

    # 변수명(key): 결과값의 타입 = Field(description="어떤 값을 넣을지 설명")
    name: str = Field(description="조회한 사람의 이름")
    yob: int = Field(description="조회한 사람의 출생 년도. 모를 경우는 -1을 대입")
    yod: int = Field(description="조회한 사람의 사망 년도. 모를 경우는 -1을 대입")
    profile: str = Field(description="조회한 사람의 주요 업적 소개")

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI

parser = PydanticOutputParser(pydantic_object=PersonInfoSchema)
# print(parser.get_format_instructions())
prompt = ChatPromptTemplate(
    messages=[
        ("system", "당신은 초등학교 역사선생님입니다. 다음 출력 형식에 맞춰 답변해주세요.\n\n<출력형식>{output_format}</출력형식>"),
        ("user", "{query}")
    ],
    partial_variables={"output_format":parser.get_format_instructions()}
)
model = ChatOpenAI(model="gpt-5.4-mini")

In [14]:
# 1. Prompt 생성
query = prompt.invoke({"query":"세종대왕에 대해 알려주세요."})
#print(query)
# 2. model에게 요청
res = model.invoke(query)
print(type(res), res)
# 3. 응답을 응답 포멧에 맞게 parsing
final_answer = parser.invoke(res)

<class 'langchain_core.messages.ai.AIMessage'> content='{"name":"세종대왕","yob":1397,"yod":1450,"profile":"조선의 제4대 임금으로, 한글을 창제하여 백성이 쉽게 글을 읽고 쓸 수 있게 했습니다. 또한 과학, 농업, 음악, 천문학 등 여러 분야의 발전을 이끌었고, 백성을 사랑하는 정치로 큰 존경을 받았습니다."}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 341, 'total_tokens': 433, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Du5ydeTCNadMBWzdxx72QJKI8yAhF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ef713-8ca5-77a1-8f41-5942142efa92-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 341, 'output_tokens': 92, 'total_tokens': 433, 'input_token_details': {'audio': 0, 'cache_read': 0},

In [16]:
print(type(final_answer))
print(final_answer)

<class '__main__.PersonInfoSchema'>
name='세종대왕' yob=1397 yod=1450 profile='조선의 제4대 임금으로, 한글을 창제하여 백성이 쉽게 글을 읽고 쓸 수 있게 했습니다. 또한 과학, 농업, 음악, 천문학 등 여러 분야의 발전을 이끌었고, 백성을 사랑하는 정치로 큰 존경을 받았습니다.'


In [17]:
print(final_answer.name)
print(final_answer.yob, final_answer.yod)
print(final_answer.profile)

세종대왕
1397 1450
조선의 제4대 임금으로, 한글을 창제하여 백성이 쉽게 글을 읽고 쓸 수 있게 했습니다. 또한 과학, 농업, 음악, 천문학 등 여러 분야의 발전을 이끌었고, 백성을 사랑하는 정치로 큰 존경을 받았습니다.


# LLM모델에 출력 형식을 설정

- ChatModel객체의 `with_structured_output(pydantic.BaseModel)` 을 이용해 모델의 출력 형식을 모델 자체에 추가할 수있다.
- `OutputParser`는 모델의 출력 결과를 받아서 형식을 변경해 준다. 그래서 Chain에 탈/부착을 통해 형식을 적용하거나 적용하지 않는 것을 자유롭게 할 수있다.
- 모델의 출력 결과를 항상 일정하게 할 경우에는 아예 **모델에 출력 형식을 설정할 수 있다.**

In [ ]:
model_with_output = model.with_structured_output(PersonInfoSchema)
res = model_with_output.invoke("세종대왕에 대해서 설명해주세요.")   # 출력형식이 설정된 모델을 호출

In [20]:
print(res.content)

{"name":"세종대왕","yob":1397,"yod":1450,"profile":"조선의 제4대 임금으로, 한글을 창제하여 백성이 쉽게 글을 읽고 쓸 수 있게 했습니다. 또한 과학, 농업, 음악, 천문학 등 여러 분야의 발전을 이끌었고, 백성을 사랑하는 정치로 큰 존경을 받았습니다."}


In [23]:
res = model_with_output.invoke("AI Agent에 특징과 장단점에 대해서 정리해서 설명해줘.")

In [27]:
print(res)

name='AI Agent' yob=-1 yod=-1 profile='AI Agent(인공지능 에이전트)는 주어진 목표를 달성하기 위해 환경을 인식하고, 판단하며, 행동을 수행하는 소프트웨어 또는 시스템입니다. 단순히 질문에 답하는 것보다 한 단계 나아가, 사용자의 의도를 바탕으로 여러 단계를 계획하고 도구를 활용해 작업을 수행할 수 있다는 점이 특징입니다.\n\n특징:\n- 목표 지향성: 특정 목표를 달성하도록 설계됩니다.\n- 자율성: 사람의 개입 없이도 일정 수준의 판단과 행동을 수행합니다.\n- 환경 인식: 입력, 상태, 데이터, 외부 시스템 정보를 바탕으로 상황을 이해합니다.\n- 계획 및 실행: 여러 단계를 순차적으로 계획하고 실행할 수 있습니다.\n- 도구 활용: 검색, 코드 실행, API 호출, DB 조회 등 외부 도구와 연동될 수 있습니다.\n- 반복 개선: 결과를 점검하고 필요하면 다시 시도하거나 전략을 수정합니다.\n\n장점:\n- 업무 자동화로 반복 작업을 줄일 수 있습니다.\n- 빠른 의사결정과 실행이 가능합니다.\n- 여러 시스템을 연결해 복합 업무를 처리할 수 있습니다.\n- 24시간 지속적으로 동작할 수 있습니다.\n- 대량의 정보를 빠르게 처리하는 데 유리합니다.\n\n단점:\n- 잘못된 판단이나 환각(hallucination)으로 오류를 낼 수 있습니다.\n- 복잡한 작업에서는 계획 실패나 비효율이 생길 수 있습니다.\n- 보안 및 개인정보 유출 위험이 있습니다.\n- 결과의 책임 소재가 불분명할 수 있습니다.\n- 고품질 성능을 위해 설계와 운영 비용이 많이 들 수 있습니다.\n\n요약하면, AI Agent는 단순 응답형 AI보다 더 능동적이고 작업 수행에 강점이 있지만, 정확성·보안·통제 측면에서 주의가 필요합니다.'


# Streaming 방식 응답 처리

- Streaming 방식 응답 처리란, LLM이 텍스트를 모두 생성할 때까지 기다리지 않고, 생성되는 즉시 **부분적인 결과**를 실시간으로 전달받아 처리하는 방식을 의미한다. 이는 사용자가 응답을 더 빠르게 인지할 수 있게 해 주며, 특히 대화형 서비스, 실시간 UI 출력, 긴 문서 생성과 같은 상황에서 매우 유용하게 활용된다.

- `invoke()` 요청으로 받는 응답은 **비 스트리밍 방식**으로 모든 응답 텍스트 생성이 완료된 이후 그 결과를 한 번에 반환하는 구조이다. 반면 Streaming 방식은 **토큰(token) 단위 또는 여러 토큰이 묶인 청크(chunk) 단위**로 연속적인 데이터 스트림을 전송한다는 점에서 큰 차이가 있다. 즉, Streaming은 마치 사람이 타이핑을 치듯이 응답을 실시간으로 “흘려보내는” 방식이라고 이해할 수 있다.

- `모델.invoke(input, config)` → 응답 데이터
    - 모델이 전체 응답을 모두 생성한 뒤, 최종 결과를 한 번에 반환하는 방식이다.
    - 배치 처리나 후처리가 중요한 경우에 적합하다.
- `모델.stream(input, config)` → generator
    - 모델이 토큰을 생성하는 즉시, 순차적으로 제공하는 generator 를 반환한다.
    - 실시간 출력, 대화형 인터페이스, 웹 스트리밍 등에 특히 적합하다.

In [28]:
model = ChatOpenAI(model="gpt-5.4-nano")
res = model.invoke("부천 맛집 세곳을 소개해주세요. 간단한 소개도 해주세요.")
print(res.content)

부천 맛집 3곳 추천드릴게요!

1) **장수삼계탕(부천점)**  
- 진한 국물의 **삼계탕**이 유명한 곳이에요. 몸보신하기 좋고, 든든하게 한 끼 해결하기 좋아요.

2) **송추가마골(부천점)**  
- 숯불향이 살아있는 **갈비/고기류**로 유명한 한식 맛집이에요. 가족 외식이나 모임에도 잘 맞아요.

3) **복성원(부천 중식당)**  
- 깔끔하면서도 깊은 맛의 **중화요리**가 강점이에요. 탕수육, 볶음면 같은 메뉴가 만족도가 높다는 후기가 많아요.

원하시면 **원하는 음식 종류(중식/한식/고기/국물)**나 **예산(1인 만원 내/2만원대 등)**, **분위기(데이트/가족/회식)** 알려주시면 그에 맞춰 더 딱 맞는 곳으로 다시 추려드릴게요!


In [31]:
gen = model.stream("부천 맛집 세곳을 소개해주세요. 간단한 소개도 해주세요.")
print(gen)

<generator object BaseChatModel.stream at 0x000001CC7492A700>


In [32]:
for token in gen:
    print(token.content, end="") # 원래는 한글자찍고 enter를 쓰는데 enter 대신 ""

물론입니다! 부천에서 가기 좋은 **맛집 3곳**을 간단히 소개해드릴게요.

1) **원조부천칼국수**
- 부천에서 유명한 칼국수 맛집으로, 담백하면서도 깊은 국물 맛이 특징입니다.  
- 점심으로 가볍게 먹기 좋고, 면발 식감도 좋아 만족도가 높아요.

2) **본가부천순대국**
- 진한 국물의 순대국과 푸짐한 건더기가 매력인 곳입니다.  
- 따뜻하게 한 그릇 먹기 좋은 메뉴 구성이라 든든하게 식사하기 좋아요.

3) **부천장어구이 (장어 전문점)**
- 신선한 장어를 숯불/불맛 느낌으로 구워내서 고소하고 풍미가 좋습니다.  
- 보양식으로 찾는 분들이 많고, 식감이 부드러워 호불호가 적어요.

원하시면 **원하시는 메뉴(고기/국밥/면/디저트)**나 **지역(중동/상동/역곡/심곡 등)** 알려주시면, 그 조건에 맞춰 더 정확히 추천해드릴게요!